# nuScenes: CPU-only 0–6 m coverage census

**NUSCENES-FULL-COVERAGE-0-6M-2026-10-28-r1**

Count available **Vehicle and Pedestrian** data before choosing training/evaluation samples. Uses the completed all-ten-part keyframe download already on Drive. **Runtime → Change runtime type → Hardware accelerator: None.** No GPU, models, training, inference, new dataset downloads or phone changes.

Reports bands **0–1, 1–2, 0–2, 2–4, 4–6, 0–6 m**, separately for frozen train, dev, held-out evaluation and previously excluded Mini recording logs. Three previously inspected evaluation logs remain in their original role and are flagged; an additional uninspected-log slice is reported. Do not add these overlapping summaries together.

Counts distinguish camera appearances, unique object/keyframe annotation observations and unique instance tracks. “Measured surface support” requires at least five conservative, exact-semantic LiDAR returns inside one unambiguous annotated 3D box in one camera/band. Annotation-centre proxies are reported separately. No ±10 cm requirement or accuracy pass/fail gate.

**Distances are camera-axis Z**, not camera Euclidean range or the nearest surface distance seen by an ultrasonic sensor. This is a coverage audit—not V0/V1 prediction accuracy or a deployment qualification.

## 1. Mount Drive and configure a separate run

Keep this RUN_ID when resuming the same code and pinned task dependencies. Never run two notebooks against the same RUN_ID at once. Existing data and completed experiments will not be overwritten. A new code/configuration identity needs a new RUN_ID.

All source archives stay on Drive. Metadata, labels, a SQLite index and only the currently active sensor archive are staged on Colab’s temporary disk. RGB headers are checked but extracted RGB pixels are not saved. Compressed result chunks go to Drive after every 64 keyframes. After a durable part completes, only its helper-owned temporary copy is removed.

In [ ]:
from google.colab import drive
from pathlib import Path
from collections import deque
import datetime, json, os, shlex, subprocess, sys

REVISION = 'NUSCENES-FULL-COVERAGE-0-6M-2026-10-28-r1'
print(REVISION, flush=True)
if os.environ.get('COLAB_GPU', '').lower() not in ('', '0', 'none'):
    raise RuntimeError('Choose Hardware accelerator: None; this workflow is CPU-only.')
drive.mount('/content/drive')
RUN_ID = 'nuscenes_full_0_6m_r1'
if not RUN_ID or any(ch not in 'abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789_-' for ch in RUN_ID):
    raise ValueError('Use a simple RUN_ID containing only letters, digits, underscore and hyphen.')
MOBILE = Path('/content/mobile_adas3d')
DATASET = Path('/content/drive/MyDrive/mobile_adas3d_datasets/nuscenes_v1_0')
OUTPUT = Path('/content/drive/MyDrive/mobile_adas3d_outputs/parking_near_field_coverage') / RUN_ID
LOCAL = Path('/content') / ('nuscenes_coverage_' + RUN_ID)
TASK_DEPS = Path('/content/nuscenes_coverage_task_deps')
PYTHON = Path(sys.executable)
WORKERS = 4
OUTPUT.mkdir(parents=True, exist_ok=True)

def run(command, name, cwd=None, env=None):
    command = [str(v) for v in command]
    stamp = datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')
    log = OUTPUT / 'colab_logs' / f'{name}_{stamp}.log'
    log.parent.mkdir(parents=True, exist_ok=True)
    merged = os.environ.copy(); merged.update(env or {})
    print('+', shlex.join(command), '\nDurable log:', log, flush=True)
    tail = deque(maxlen=30)
    with log.open('w') as stream:
        process = subprocess.Popen(command, cwd=cwd, env=merged, stdout=subprocess.PIPE,
                                   stderr=subprocess.STDOUT, text=True, bufsize=1)
        try:
            for line in process.stdout:
                print(line, end='', flush=True); stream.write(line); stream.flush(); tail.append(line.rstrip())
        finally:
            process.stdout.close()
        status = process.wait(); stream.write(f'\nEXIT CODE: {status}\n'); stream.flush()
    if status:
        raise RuntimeError(f'Exit {status}; preserve files and inspect {log}\n' + '\n'.join(tail))
    return log

print('Drive dataset:', DATASET, '\nResult directory:', OUTPUT, '\nTemporary scratch:', LOCAL)

## 2. Pull main, install CPU task dependencies, and run tests

This does not replace Colab’s Torch/CUDA installation. The four pinned task packages are installed in their own import directory and used only by subprocesses. No PyTorch, nuScenes SDK, CUDA compiler or old M62/M67 runtime is needed. An unexpected Git branch or tracked edit stops setup rather than discarding changes.

In [ ]:
url = '/'.join(['https:', '', 'github.com', 'Ali-RT', 'mobile_adas3d.git'])
if not MOBILE.exists():
    run(['git', 'clone', '--branch', 'main', url, MOBILE], 'clone_mobile')
branch = subprocess.check_output(['git', 'branch', '--show-current'], cwd=MOBILE, text=True).strip()
if branch != 'main':
    raise RuntimeError(f'Preserve this checkout; expected main, found {branch!r}. Use a clean main checkout.')
changes = subprocess.check_output(['git', 'status', '--porcelain', '--untracked-files=no'], cwd=MOBILE, text=True)
if changes.strip():
    raise RuntimeError('Tracked project changes exist. Preserve them and use a clean checkout; no reset is performed.')
run(['git', 'pull', '--ff-only'], 'update_mobile', MOBILE)
SCRIPT = MOBILE / 'scripts/audit_nuscenes_coverage_0_6m.py'
required = [SCRIPT, MOBILE/'scripts/nuscenes_coverage_data.py', MOBILE/'scripts/parking_near_field_geometry.py',
            MOBILE/'configs/nuscenes_parking_frozen_roles_v1.json', MOBILE/'tests/test_nuscenes_full_coverage.py',
            DATASET/'download_manifest_keyframes_all.json']
for path in required:
    if not path.is_file():
        raise FileNotFoundError(f'Missing required input: {path}. No coverage processing has started.')
config = json.loads((MOBILE/'configs/nuscenes_coverage_0_6m_v1.json').read_text())
if config['revision'] != REVISION:
    raise RuntimeError('Notebook/code revision differs. Reload the latest notebook before running.')
requirements = [name+'=='+version for name, version in config['runtime_packages'].items()]
run([PYTHON, '-m', 'pip', 'install', '--quiet', '--upgrade', '--only-binary=:all:', '--no-deps',
     '--target', TASK_DEPS, *requirements], 'cpu_dependencies', MOBILE)
TASK_ENV = {'PYTHONPATH': os.pathsep.join([str(TASK_DEPS), str(MOBILE)]),
            'CUDA_VISIBLE_DEVICES': '', 'OMP_NUM_THREADS': '1', 'OPENBLAS_NUM_THREADS': '1',
            'MKL_NUM_THREADS': '1', 'PYTHONDONTWRITEBYTECODE': '1'}
run([PYTHON, '-m', 'unittest', 'discover', '-s', 'tests', '-p', 'test_nuscenes_full_coverage.py', '-v'],
    'coverage_tests', MOBILE, TASK_ENV)

def stage(name):
    return run([PYTHON, '-X', 'faulthandler', '-u', SCRIPT, name, '--dataset', DATASET,
                '--output', OUTPUT, '--local', LOCAL, '--workers', str(WORKERS)],
               'coverage_'+name, MOBILE, TASK_ENV)

READY = True
print('CPU task setup and synthetic tests passed. No sensor archive has been processed yet.')

## 3. Verify the reviewed download and prepare the local metadata index

Checks the exact returned download manifest, archive sizes/receipts and frozen scene/log map. Local copies are SHA256-checked against those receipts. Metadata tables are streamed into SQLite instead of loading the full dataset into RAM. Semantic labels are extracted locally once. This step can take time on Drive; the progress/log remains visible. It does not require a GPU.

Local disk checks run before copying/extracting each archive, leaving an 8 GB reserve. If this cell fails for local disk space, preserve Drive data and choose a Colab CPU runtime with more temporary disk. It will not fill your Mac.

In [ ]:
if not READY:
    raise RuntimeError('Run sections 1 and 2 in order first.')
stage('prepare')

## 4. Run all ten parts, with resumable chunk checkpoints

Leave the CPU runtime connected. A part is copied from Drive, checked, its six camera headers inspected, and its local LiDAR scans processed with four workers. No image forest is written to Drive. Progress reports the keyframes durably saved.

If Colab disconnects or resets: reopen this notebook and rerun from the top with the **same RUN_ID**. Completed parts are verified and skipped; a partially completed part restages its source and resumes from its saved 64-keyframe chunks. Incomplete in-memory work may need repeating. Never delete the source archives, frozen role map or completed results to get past a mismatch.

In [ ]:
if not READY:
    raise RuntimeError('Run sections 1 and 2 in order first.')
stage('run')
summary = json.loads((OUTPUT/'coverage_summary.json').read_text())
assert summary['complete'] and summary['unique_keyframes'] == 34149 and summary['unique_scenes'] == 850
assert not summary['accuracy_pass_fail_enabled'] and not summary['training_performed']
print('Complete:', summary['unique_keyframes'], 'keyframes /', summary['unique_scenes'], 'scenes')
for group, result in summary['groups'].items():
    print('\n' + group, '—', result['keyframes'], 'keyframes')
    for cls in ('Vehicle', 'Pedestrian'):
        for band in ('0-1m', '1-2m', '0-2m', '2-4m', '4-6m', '0-6m'):
            cell = result['cells'][f'{cls}:{band}']
            print(cls, band, '| surface-supported observations:', cell['surface_supported_annotation_observations'],
                  '| unique tracks:', cell['surface_supported_instance_tracks'],
                  '| centre-proxy observations:', cell['centre_in_image_annotation_observations'])

## 5. Package the small results ZIP and download it

Return **nuscenes_0_6m_coverage_results.zip** for review. It contains the summary, per-scene/per-recording-log counts, frozen run identity and part/chunk receipts—not raw images, LiDAR or model weights. Detailed compressed chunks remain on Drive for reproducibility. These results guide dataset selection and KPI design; they do not yet measure V0/V1 model performance.

In [ ]:
stage('bundle')
BUNDLE = OUTPUT/'nuscenes_0_6m_coverage_results.zip'
print('Saved:', BUNDLE)
from google.colab import files
files.download(str(BUNDLE))